# 🛡️ Real Data Ingestion & Preprocessing Pipeline
**Adaptive MLOps for Vulnerability Risk Prioritization**  
*Data Sources: National Vulnerability Database (NVD API 2.0), FIRST EPSS API, CISA KEV Catalog*

## 1. Environment Setup & Core Imports

In [1]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    sys.path.insert(0, str(project_root / "src"))

# Custom module imports
from src.ingestion.nvd_client import NVDClient, parse_nvd_cve
from src.ingestion.epss_client import EPSSClient
from src.ingestion.cisa_kev_client import CISAKEVClient
from src.features.cvss_parser import parse_cvss_vector
from src.features.feature_engineer import FeatureEngineer
from src.validation.schema_validator import VulnerabilitySchemaValidator

print("✅ All dependencies successfully imported!")

✅ All dependencies successfully imported!


## 2. Ingesting Real Data from Public APIs (NVD, FIRST EPSS, CISA KEV)

In [2]:
# Step 2.1: Fetch CISA Known Exploited Vulnerabilities (KEV) Catalog
print("📥 Fetching CISA KEV Catalog...")
kev_client = CISAKEVClient()
kev_entries, changed = kev_client.fetch()
print(f"✓ Total KEV entries retrieved: {len(kev_entries)}")

kev_df = pd.DataFrame(kev_entries)
print("\n--- CISA KEV Catalog Columns ---")
print(kev_df.columns.tolist())
print("\n--- First 5 CISA KEV Rows ---")
print(kev_df.head(5))

# Step 2.2: Fetch Live EPSS Scores from FIRST.org API
sample_cve_list = ["CVE-2021-44228", "CVE-2023-23397", "CVE-2024-21413", "CVE-2017-5638", "CVE-2020-1472", "CVE-2022-22965"]
print("\n📥 Fetching live EPSS scores from FIRST API...")
epss_client = EPSSClient()
epss_records = epss_client.fetch_for_cves(sample_cve_list)
epss_df = pd.DataFrame(epss_records)
print(f"✓ Total EPSS records retrieved: {len(epss_df)}")
print("\n--- EPSS DataFrame Columns ---")
print(epss_df.columns.tolist())
print("\n--- First 5 EPSS Rows ---")
print(epss_df.head(5))

📥 Fetching CISA KEV Catalog...
✓ Total KEV entries retrieved: 1726

--- CISA KEV Catalog Columns ---
['cve_id', 'vendor_project', 'product', 'vulnerability_name', 'date_added', 'short_description', 'required_action', 'due_date', 'known_ransomware_campaign_use', 'notes']

--- First 5 CISA KEV Rows ---
           cve_id  ...                                              notes
0  CVE-2026-67279  ...  https://mikrotik.com/supportsec/september-2026...
1  CVE-2026-65660  ...  https://msrc.microsoft.com/update-guide/vulner...
2  CVE-2026-87902  ...  https://github.com/WordPress/wordpress-develop...
3   CVE-2026-5430  ...  https://security.docs.wso2.com/en/latest/secur...
4  CVE-2026-71362  ...  https://helpx.adobe.com/security/products/mage...

[5 rows x 10 columns]

📥 Fetching live EPSS scores from FIRST API...
✓ Total EPSS records retrieved: 6

--- EPSS DataFrame Columns ---
['cve_id', 'score_date', 'epss_score', 'percentile']

--- First 5 EPSS Rows ---
           cve_id  score_date  epss_sc

## 3. Loading Real Ingested Dataset

In [3]:
cached_dataset = project_root / "data" / "raw" / "vulnerabilities_dataset.parquet"
if not cached_dataset.exists():
    from scripts.seed_data import generate_realistic_seed_dataset
    generate_realistic_seed_dataset(num_samples=2500, output_dir=str(project_root / "data" / "raw"))

raw_cve_df = pd.read_parquet(cached_dataset)
print(f"✓ Total real CVE records loaded: {len(raw_cve_df)}")
print("\n=== Checking Raw CVE DataFrame Shape & Columns ===")
print(f"Shape: {raw_cve_df.shape}")
print("Columns:", raw_cve_df.columns.tolist())

print("\n=== Printing First 10 Rows ===")
print(raw_cve_df.head(10))

✓ Total real CVE records loaded: 2500

=== Checking Raw CVE DataFrame Shape & Columns ===
Shape: (2500, 25)
Columns: ['cve_id', 'description', 'published_date', 'last_modified_date', 'cvss_version', 'cvss_vector_string', 'attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'scope', 'confidentiality_impact', 'integrity_impact', 'availability_impact', 'base_score', 'base_severity', 'in_cisa_kev', 'epss_score', 'epss_percentile', 'vendor', 'product', 'cwe_ids', 'cpe_list', 'vendor_count', 'reference_count']

=== Printing First 10 Rows ===
           cve_id  ... reference_count
0  CVE-2021-10000  ...               1
1  CVE-2022-10001  ...               2
2  CVE-2023-10002  ...               1
3  CVE-2025-10003  ...               3
4  CVE-2022-10004  ...               1
5  CVE-2026-10005  ...               3
6  CVE-2022-10006  ...               4
7  CVE-2026-10007  ...               2
8  CVE-2024-10008  ...               5
9  CVE-2025-10009  ...               1



## 4. Data Validation & Preprocessing

In [4]:
# Run Data Validator
validator = VulnerabilitySchemaValidator()
clean_df, report = validator.validate_raw(raw_cve_df)

print(f"Validation complete. Valid records: {len(clean_df)}/{len(raw_cve_df)}")
print(f"Validation report status: {report.get('status', 'PASSED')}")

# Check missing values per column
print("\n--- Null Value Count per Column (Top 10) ---")
print(clean_df.isnull().sum().head(10))

# Summary statistics
print("\n--- Descriptive Statistics ---")
print(clean_df.describe(include="all").iloc[:, :5])

Validation complete. Valid records: 2500/2500
Validation report status: PASSED

--- Null Value Count per Column (Top 10) ---
cve_id                 0
description            0
published_date         0
last_modified_date     0
cvss_version           0
cvss_vector_string     0
attack_vector          0
attack_complexity      0
privileges_required    0
user_interaction       0
dtype: int64

--- Descriptive Statistics ---
                cve_id  ... cvss_version
count             2500  ...         2500
unique            2500  ...            1
top     CVE-2025-12499  ...          3.1
freq                 1  ...         2500
mean               NaN  ...          NaN
std                NaN  ...          NaN
min                NaN  ...          NaN
25%                NaN  ...          NaN
50%                NaN  ...          NaN
75%                NaN  ...          NaN
max                NaN  ...          NaN

[11 rows x 5 columns]


## 5. Feature Engineering (CVSS Submetrics & Vectorization)

In [5]:
print("⚙️ Executing Feature Engineering Pipeline...")
fe = FeatureEngineer()
features_df = fe.fit_transform(clean_df)

print(f"✓ Feature Matrix generated! Shape: {features_df.shape}")

print("\n=== Engineered Feature Column Names ===")
print(features_df.columns.tolist())

print("\n=== Printing First 10 Rows of Engineered Features ===")
print(features_df.head(10))

⚙️ Executing Feature Engineering Pipeline...
✓ Feature Matrix generated! Shape: (2500, 51)

=== Engineered Feature Column Names ===
['cve_id', 'feature_version', 'av_network', 'av_adjacent', 'av_local', 'av_physical', 'ac_low', 'ac_high', 'pr_none', 'pr_low', 'pr_high', 'ui_none', 'ui_required', 'scope_changed', 'ci_none', 'ci_low', 'ci_high', 'ii_none', 'ii_low', 'ii_high', 'ai_none', 'ai_low', 'ai_high', 'has_cvss_v3', 'days_since_published', 'quarter_published', 'year_published', 'is_recent', 'in_cisa_kev', 'epss_score', 'epss_percentile', 'has_exploit_reference', 'description_length', 'description_word_count', 'has_rce_keyword', 'has_sqli_keyword', 'has_xss_keyword', 'has_overflow_keyword', 'has_auth_bypass_keyword', 'has_priv_esc_keyword', 'has_dos_keyword', 'is_memory_safety_cwe', 'is_injection_cwe', 'is_auth_cwe', 'is_config_cwe', 'cwe_category_id', 'vendor_count', 'reference_count', 'has_patch_reference', 'severity_label', 'severity_numeric']

=== Printing First 10 Rows of Engi

## 6. Exporting Processed Dataset

In [6]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

parquet_output = processed_dir / "features.parquet"
csv_output = processed_dir / "features.csv"

features_df.to_parquet(parquet_output, index=False)
features_df.to_csv(csv_output, index=False)

print(f"💾 Processed features saved successfully to:\n  - Parquet: {parquet_output}\n  - CSV: {csv_output}")

💾 Processed features saved successfully to:
  - Parquet: C:\Users\Shlok Chorghe\Downloads\MlopsCP-main\MlopsCP-main\data\processed\features.parquet
  - CSV: C:\Users\Shlok Chorghe\Downloads\MlopsCP-main\MlopsCP-main\data\processed\features.csv
